# Turkish Constitutional Court decision classifier (cased BERTurk)

**Project:** faithfulness comparison of XAI methods in high-stakes Turkish text classification.

**This notebook:** fine-tunes `dbmdz/bert-base-turkish-128k-cased` on
`icgcihan/Turkish_Constutional_Court_Decisions` for binary violation / no-violation
classification, and saves the model and an example pool for the explanation analysis.

### Before running (Kaggle right panel):
1. **Settings > Accelerator > GPU (T4 x1 or P100)**
2. **Settings > Internet > ON** (the dataset and model are downloaded)
3. Run All. Takes ~30-60 min.

Outputs are under `/kaggle/working/`; download them from the **Output** tab when done:
`model/`, `test_predictions.csv`, `example_pool.json`, `metrics.json`.

In [ ]:
# 1) environment (most is preinstalled on Kaggle; we pin versions)
!pip -q install -U "transformers>=4.44" "datasets>=2.20" scikit-learn 2>/dev/null
import transformers, datasets, torch
print("transformers", transformers.__version__, "| datasets", datasets.__version__,
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 2) config + reproducibility
import numpy as np, random, os, json
from transformers import set_seed
SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)

DATASET   = "icgcihan/Turkish_Constutional_Court_Decisions"
MODEL     = "dbmdz/bert-base-turkish-128k-cased"
TEXT_COL  = "text"
LABEL_COL = "labels"
MAX_LEN   = 512          # head truncation
EPOCHS    = 3
LR        = 2e-5
BATCH     = 8            # safe for 128k vocab + 512 len; effective 16 with grad_accum
GRAD_ACC  = 2
OUTDIR    = "/kaggle/working"


In [ ]:
# 3) data
from datasets import load_dataset
ds = load_dataset(DATASET)
print({k: len(ds[k]) for k in ds})
from collections import Counter
for k in ds:
    print(k, dict(Counter(ds[k][LABEL_COL])))
# labels are already 0/1 ints
NUM_LABELS = 2

In [ ]:
# 4) tokenization (head-512, cased, diacritics preserved)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.truncation_side == "right", "truncation_side must be 'right' for head truncation"

def prep(batch):
    enc = tok(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)
    enc["labels"] = batch[LABEL_COL]
    return enc

keep = ds["train"].column_names
ds_tok = ds.map(prep, batched=True, remove_columns=keep)
print(ds_tok["train"][0].keys())

In [ ]:
# 5) model + metrics
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import f1_score, precision_recall_fscore_support, average_precision_score, confusion_matrix
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)
collator = DataCollatorWithPadding(tok)

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits); preds = probs.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0,1], zero_division=0)
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_pos": f1[1], "f1_neg": f1[0],
        "prec_pos": p[1], "rec_pos": r[1],
        "pr_auc_pos": average_precision_score(labels, probs[:,1]),
    }

In [ ]:
# 6) training (version-compatible TrainingArguments)
import inspect
from transformers import TrainingArguments, Trainer

_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
_eval_key = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"
args_kw = dict(
    output_dir=f"{OUTDIR}/ckpt",
    learning_rate=LR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=16,
    gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01, warmup_ratio=0.1,
    logging_steps=50, save_total_limit=1, seed=SEED,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    report_to="none",
)
args_kw[_eval_key] = "epoch"; args_kw["save_strategy"] = "epoch"
args = TrainingArguments(**args_kw)

trainer = Trainer(model=model, args=args,
                  train_dataset=ds_tok["train"], eval_dataset=ds_tok["validation"],
                  processing_class=tok, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
# 7) test evaluation + leakage check
test_out = trainer.predict(ds_tok["test"])
metrics = {k.replace("test_",""): float(v) for k,v in test_out.metrics.items() if isinstance(v,(int,float))}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

probs = softmax(test_out.predictions); preds = probs.argmax(-1); labels = np.array(test_out.label_ids)
print("Confusion matrix [0,1]:\n", confusion_matrix(labels, preds, labels=[0,1]))

if metrics.get("test_f1_macro", metrics.get("f1_macro",0)) > 0.97:
    print("\nWARNING: F1 is very high; possible label leakage. The court ruling may remain in the texts, check it.")
else:
    print("\nF1 is in a reasonable range, consistent with a leakage-safe predict-from-the-facts task.")

In [ ]:
# 8) save: model + test predictions + example pool
import pandas as pd
os.makedirs(f"{OUTDIR}/model", exist_ok=True)
trainer.save_model(f"{OUTDIR}/model"); tok.save_pretrained(f"{OUTDIR}/model")

conf = probs.max(1)
df = pd.DataFrame({
    "idx": np.arange(len(labels)), "true": labels, "pred": preds,
    "prob_1": probs[:,1], "confidence": conf, "correct": (preds==labels),
    "text": ds["test"][TEXT_COL], "Haklar": ds["test"]["Haklar"],
})
df.to_csv(f"{OUTDIR}/test_predictions.csv", index=False)

# qualitative pool: high-confidence correct instances, balanced per class (~25+25)
pool = (df[df.correct]
        .sort_values("confidence", ascending=False)
        .groupby("true", group_keys=False).head(25))
pool[["idx","true","pred","confidence"]].to_json(
    f"{OUTDIR}/example_pool.json", orient="records", force_ascii=False, indent=2)

json.dump(metrics, open(f"{OUTDIR}/metrics.json","w"), ensure_ascii=False, indent=2)
print("Saved:", os.listdir(OUTDIR))
print(f"Example pool: {len(pool)} instances (~25 per class).")

## Done: files to download (Output tab)
- **`model/`** the fine-tuned model used locally for the explanation analysis (~700 MB).
- **`test_predictions.csv`** all test predictions + confidence scores.
- **`example_pool.json`** ~50 high-confidence correct instances for the qualitative heatmaps.
- **`metrics.json`** F1-macro, PR-AUC, per-class metrics.

**Next:** place these outputs under `models/` and `results/`, then run the explanation methods (attention / rollout / IG / relevance).